# V0.2 — Deployment-Valid Phone-Only GRU (SIH26168)

**Question:** Can a small causal GRU learn a useful velocity residual using only
smartphone-available information, and does it **retain** that improvement during a
fully recursive GNSS-denied rollout?

**Why V0.2:** The V0.1 recursive audit (CASE B) showed the V0 16-feature checkpoint
is NOT deployment-valid — it read vehicle-CAN/GNSS reference channels during the
simulated blackout and its recursive benefit collapsed to +8.6% avg.

**This notebook (GPU):**
1. Clones the `orbit-inertia` code from GitHub (code only, no dataset).
2. Trains a 1-layer GRU (hidden=32, ~4.2k params) from scratch on **9 phone-only
   features**: `accel_x/y/z`, `gravity_x/y/z`, `gyro_pitch` (measured phone IMU),
   `nav_speed`, `nav_heading` (internal A0 classical DR state).
3. Evaluates under a **FULLY RECURSIVE** rollout (same 39 blackout windows,
   10/30/60/120 s) vs the classical A0 baseline, plus an **oracle-state
   diagnostic** (reference state fed into nav channels — DIAGNOSTIC ONLY, never
   headlined).

**Setup:** Internet ON, accel **GPU T4**. Dataset: upload `S4_synced.csv` as a
Kaggle Dataset (see step 2).

## Step 1 — Clone code from GitHub

In [ ]:
!git clone --depth 1 https://github.com/nikhilwankhedee/orbit-inertia.git
print('Cloned orbit-inertia.')

## Step 2 — Verify the dataset mount

Upload `processed/S4_synced.csv` as a Kaggle Dataset (e.g. named `sih26168`). The
cell below locates it automatically and picks a `DATA_ROOT`. Edit only if it fails.

In [ ]:
import glob

# Find the synced dataset anywhere under /kaggle/input
matches = glob.glob('/kaggle/input/**/S4_synced.csv', recursive=True)
print('S4_synced.csv found under /kaggle/input:')
for m in matches:
    print('  ', m)

if matches:
    # run_all accepts either the dataset dir or the CSV path itself
    DATA_ROOT = matches[0]
else:
    DATA_ROOT = '/kaggle/input/sih26168'  # fallback; edit to your mount
print()
print('DATA_ROOT =', DATA_ROOT)

## Step 3 — Train + fully recursive evaluate (one shot)

`run_all.py --variant v02` runs the official V0.2 pipeline:

1. **Train** `train_velocity_residual_gru_v02.py` — 200 epochs, Adam lr=1e-3,
   MSE, early stopping, causal window 20 samples (~2 s @ 10 Hz), segment split
   Train Seg0+Seg1 / Val Seg2 / Test Seg3, z-score normalization fit on TRAIN only
   (seed 42).
2. **Evaluate** `evaluate_velocity_residual_gru_v02.py` — fully recursive rollout
   (primary) + oracle-state (diagnostic only), all reports + plots.

No training data is read in this cell; the dataset stays mounted under `DATA_ROOT`.

In [ ]:
!python orbit-inertia/sih26168/src/run_all.py \
    --data-root {DATA_ROOT} \
    --variant v02 \
    --epochs 200 \
    --batch-size 256 \
    --hidden-size 32 \
    --context-len 20 \
    --stride 5 \
    --seed 42

## Step 4 — Verify outputs

Everything lands in `orbit-inertia/sih26168/outputs/ml_v02/`:

- `best_model.pt`, `normalization.npz`, `train_config.json`, `training_log.csv`
- `training_report.txt`, `recursive_evaluation_report.txt`, `v02_report.txt` (15-item)
- `recursive_comparison.csv` (per-window A0 / V0.2 / Oracle metrics)
- `plots/v02_position_error_{10,30,60,120}s.png`, `v02_final_error.png`,
  `v02_three_way_mae.png`, `v02_correction_magnitude.png`, `v02_pred_vs_target.png`

In [ ]:
import os

out = 'orbit-inertia/sih26168/outputs/ml_v02'
expected = [
    'best_model.pt', 'normalization.npz', 'train_config.json', 'training_log.csv',
    'training_report.txt', 'recursive_evaluation_report.txt', 'v02_report.txt',
    'recursive_comparison.csv',
]
for f in expected:
    p = os.path.join(out, f)
    print(('[OK]   ' if os.path.exists(p) else '[MISS] ') + f)
for d in ['v02_position_error_10s.png', 'v02_position_error_30s.png',
          'v02_position_error_60s.png', 'v02_position_error_120s.png',
          'v02_final_error.png', 'v02_three_way_mae.png',
          'v02_correction_magnitude.png', 'v02_pred_vs_target.png']:
    p = os.path.join(out, 'plots', d)
    print(('[OK]   ' if os.path.exists(p) else '[MISS] ') + 'plots/' + d)

## Step 5 — Result summary

Prints the recursive A0 vs V0.2 table and the CASE verdict. **Recursive is the
primary result**; oracle-state is diagnostic only.

In [ ]:
report = os.path.join(out, 'recursive_evaluation_report.txt')
print(open(report).read() if os.path.exists(report) else 'Report not found.')
print()
print('=' * 70)
print('VERDICT (from v02_report.txt, section 15):')
v02r = os.path.join(out, 'v02_report.txt')
if os.path.exists(v02r):
    for line in open(v02r):
        if 'CASE' in line:
            print(line.rstrip())

## Step 6 — Download artifacts to local

Click **Output → Download all** (top-right of this notebook). Store the folder as
`sih26168/outputs/ml_v02/` locally.